# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [336]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [337]:
# Load datasets
import csv
aviation_df = pd.read_csv('data/AviationData.csv', encoding='latin1', low_memory=False)
state_codes_df = pd.read_csv('data/USState_Codes.csv')

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [338]:
# Preview data
aviation_df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [339]:
# Check shape (rows, columns)
aviation_df.shape

(88889, 31)

In [340]:
# Column names
aviation_df.columns

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='object')

In [341]:
# Info: datatypes + missing values
aviation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

In [342]:
# Count missing values per column
aviation_df.isna().sum().sort_values(ascending=False)

Schedule                  76307
Air.carrier               72241
FAR.Description           56866
Aircraft.Category         56602
Longitude                 54516
Latitude                  54507
Airport.Code              38757
Airport.Name              36185
Broad.phase.of.flight     27165
Publication.Date          13771
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Engine.Type                7096
Report.Status              6384
Purpose.of.flight          6192
Number.of.Engines          6084
Total.Uninjured            5912
Weather.Condition          4492
Aircraft.damage            3194
Registration.Number        1382
Injury.Severity            1000
Country                     226
Amateur.Built               102
Model                        92
Make                         63
Location                     52
Investigation.Type            0
Event.Date                    0
Accident.Number               0
Event.Id                      0
dtype: i

In [343]:
#For numerical columns:
aviation_df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [344]:
aviation_df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [345]:
#For categorical columns:
aviation_df.describe(include='object')

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Amateur.Built,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
count,88889,88889,88889,88889,88837,88663,34382,34373,50132,52704,...,88787,81793,32023,12582,82697,16648,84397,61724,82505,75118
unique,87951,2,88863,14782,27758,219,25589,27154,10374,24870,...,2,12,31,3,26,13590,4,12,17074,2924
top,20001212X19172,Accident,CEN22LA149,1984-06-30,"ANCHORAGE, AK",United States,332739N,0112457W,NONE,Private,...,No,Reciprocating,091,NSCH,Personal,Pilot,VMC,Landing,Probable Cause,25-09-2020
freq,3,85015,2,25,434,82248,19,24,1488,240,...,80312,69530,18221,4474,49448,258,77303,15428,61754,17019


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [346]:
#Create a New Feature
#important for your analysis + grading

aviation_df['Total.Injuries'] = (
    aviation_df['Total.Fatal.Injuries'] +
    aviation_df['Total.Serious.Injuries'] +
    aviation_df['Total.Minor.Injuries']
)

In [347]:
#Standardize Categorical Columns

aviation_df['Weather.Condition'] = aviation_df['Weather.Condition'].str.upper()
aviation_df['Broad.phase.of.flight'] = aviation_df['Broad.phase.of.flight'].str.title()
aviation_df['Make'] = aviation_df['Make'].str.upper()

In [348]:
#Aircraft Category
aviation_df['Aircraft.Category'].value_counts(dropna=False)

Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

In [349]:
#Aircraft.Category
#Keep "Airplane"
#Drop everything else including NaN (Other categories helicopter, glider, etc. are irrelevant to the client)
aviation_df = aviation_df[aviation_df['Aircraft.Category'] == 'Airplane']
aviation_df['Aircraft.Category'].value_counts(dropna=False)


Aircraft.Category
Airplane    27617
Name: count, dtype: int64

In [350]:
#Amature built
aviation_df['Amateur.Built'].value_counts(dropna=False)

Amateur.Built
No     24417
Yes     3183
NaN       17
Name: count, dtype: int64

In [351]:
#only want professionally built aircraft
#remove amateur and NaN
aviation_df = aviation_df[aviation_df['Amateur.Built'] == 'No']


#check if all amateur bult removed and NaN - "No" for professional
aviation_df['Amateur.Built'].value_counts(dropna=False)

Amateur.Built
No    24417
Name: count, dtype: int64

In [352]:
#make sure it's datetime
aviation_df['Event.Date'] = pd.to_datetime(aviation_df['Event.Date'], errors='coerce')


#Filter date for the last 40 years
from datetime import datetime

from datetime import datetime

cutoff_date = pd.Timestamp(datetime.now() - pd.DateOffset(years=40))
aviation_df = aviation_df[aviation_df['Event.Date'] >= cutoff_date]

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [353]:
# Clean Injury Columns
# Replace missing values with 0
injury_cols = [
    'Total.Fatal.Injuries',
    'Total.Serious.Injuries',
    'Total.Minor.Injuries',
    'Total.Uninjured'
]

aviation_df[injury_cols] = aviation_df[injury_cols].fillna(0)
# Preview data
aviation_df.head()




,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.Injuries
14711,20001213X33697,Accident,SEA86FA121B,1986-05-19,"MEAD, WA",United States,NaN,NaN,NaN,NaN,...,NaN,2.0,0.0,1.0,0.0,VMC,Descent,Probable Cause,17-10-2016,NaN
14712,20001213X33697,Accident,SEA86FA121A,1986-05-19,"MEAD, WA",United States,NaN,NaN,NaN,NaN,...,NaN,2.0,0.0,1.0,0.0,VMC,Maneuvering,Probable Cause,17-10-2016,NaN
14805,20001213X33489,Incident,BFO86IA031B,1986-05-30,"CALVERTON, NY",United States,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,69.0,VMC,Descent,Probable Cause,09-10-2018,NaN
14806,20001213X33489,Incident,BFO86IA031A,1986-05-30,"CALVERTON, NY",United States,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,69.0,VMC,Descent,Probable Cause,09-10-2018,NaN
16646,20001213X30060,Accident,DCA87MA018B,1987-01-15,"KEARNS, UT",United States,NaN,NaN,NaN,NaN,...,NaN,10.0,0.0,0.0,0.0,VMC,Maneuvering,Probable Cause,10-07-2019,NaN


In [354]:
#To Estimate Total Passengers
aviation_df['Total.Passengers'] = (
    aviation_df['Total.Fatal.Injuries'] +
    aviation_df['Total.Serious.Injuries'] +
    aviation_df['Total.Minor.Injuries'] +
    aviation_df['Total.Uninjured']
)

In [355]:
#clean rows that may have 0 passengers, leads to division errors
aviation_df = aviation_df[aviation_df['Total.Passengers'] > 0]

In [356]:
#Create Injury Severity Metric
#Fatal + Serious Injuries
aviation_df['Serious.Fatal.Injuries'] = (
    aviation_df['Total.Fatal.Injuries'] +
    aviation_df['Total.Serious.Injuries']
)

#Injury Rate
aviation_df['Injury.Rate'] = (
    aviation_df['Serious.Fatal.Injuries'] / aviation_df['Total.Passengers']
)



In [357]:
# Cleaning Aircraft Damage Column
# Inspect values
aviation_df['Aircraft.damage'].value_counts(dropna=False)

Aircraft.damage
Substantial    16812
Destroyed       2257
NaN              785
Minor            588
Unknown           74
Name: count, dtype: int64

In [358]:
# standardize to avoid hidden inconsistencies
aviation_df['Aircraft.damage'] = aviation_df['Aircraft.damage'].str.title().str.strip()

In [359]:
#Handling missing values
#You have only ~785 missing values, small enough to drop safely.
aviation_df = aviation_df.dropna(subset=['Aircraft.damage'])

In [360]:
#Creating “Destroyed vs Not”
aviation_df['Is.Destroyed'] = aviation_df['Aircraft.damage'].apply(
    lambda x: 1 if x == 'Destroyed' else 0
)

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [361]:
#Inspect the Column
aviation_df['Make'].value_counts().head(20)

Make
CESSNA                6968
PIPER                 3926
BEECH                 1393
BOEING                 484
MOONEY                 357
AIR TRACTOR INC        219
BELLANCA               218
CIRRUS DESIGN CORP     217
MAULE                  215
AIR TRACTOR            203
AERONCA                200
CHAMPION               157
GRUMMAN                143
LUSCOMBE               139
STINSON                129
CIRRUS                 128
NORTH AMERICAN         104
TAYLORCRAFT             93
DEHAVILLAND             92
AERO COMMANDER          88
Name: count, dtype: int64

In [362]:
#Standardize Text
aviation_df['Make'] = aviation_df['Make'].str.upper().str.strip()

#remove missing values
aviation_df = aviation_df.dropna(subset=['Make'])

In [363]:
#Filter by Threshold (≥ 50 Records)
#First count occurrences

make_counts = aviation_df['Make'].value_counts()

#keeping only those ≥ 50
valid_makes = make_counts[make_counts >= 50].index

aviation_df = aviation_df[aviation_df['Make'].isin(valid_makes)]

#Show results
aviation_df['Make'].value_counts().head(10)

Make
CESSNA                6968
PIPER                 3926
BEECH                 1393
BOEING                 484
MOONEY                 357
AIR TRACTOR INC        219
BELLANCA               218
CIRRUS DESIGN CORP     217
MAULE                  215
AIR TRACTOR            203
Name: count, dtype: int64

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [364]:
#Inspecting the Model Column
aviation_df['Model'].value_counts().head(20)

Model
172          745
152          307
182          299
172S         267
PA28         263
172N         244
SR22         232
180          213
A36          179
172M         178
PA-18-150    174
150          172
PA-28-140    165
172P         141
140          115
737          113
170B         107
172R         106
PA-28-180    105
PA-28-161    102
Name: count, dtype: int64

In [365]:
#Remove NaNs
aviation_df = aviation_df.dropna(subset=['Model'])

#Standardize the Model Column
aviation_df['Model'] = aviation_df['Model'].str.upper().str.strip()

#Inspect Make and Model Together
aviation_df[['Make', 'Model']].value_counts().head(20)


Make                Model    
CESSNA              172          745
                    152          307
                    182          299
                    172S         267
PIPER               PA28         263
CESSNA              172N         244
                    180          213
                    172M         178
PIPER               PA-18-150    174
CESSNA              150          172
PIPER               PA-28-140    165
BEECH               A36          162
CIRRUS DESIGN CORP  SR22         142
CESSNA              172P         141
                    140          115
BOEING              737          113
CESSNA              170B         107
                    172R         106
PIPER               PA-28-180    105
                    PA-28-161    102
Name: count, dtype: int64

In [366]:
#Create a Unique Aircraft Identifier
aviation_df['Aircraft.Type'] = (
    aviation_df['Make'] + ' ' + aviation_df['Model']
)

#Inspect the New Column
aviation_df['Aircraft.Type'].value_counts().head(20)


Aircraft.Type
CESSNA 172                 745
CESSNA 152                 307
CESSNA 182                 299
CESSNA 172S                267
PIPER PA28                 263
CESSNA 172N                244
CESSNA 180                 213
CESSNA 172M                178
PIPER PA-18-150            174
CESSNA 150                 172
PIPER PA-28-140            165
BEECH A36                  162
CIRRUS DESIGN CORP SR22    142
CESSNA 172P                141
CESSNA 140                 115
BOEING 737                 113
CESSNA 170B                107
CESSNA 172R                106
PIPER PA-28-180            105
PIPER PA-28-161            102
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [367]:
#Inspecting all columns
cols = [
    'Engine.Type',
    'Weather.Condition',
    'Number.of.Engines',
    'Purpose.of.flight',
    'Broad.phase.of.flight'
]

for col in cols:
    print("\n", col)
    print(aviation_df[col].value_counts(dropna=False).head(15))


 Engine.Type
Engine.Type
Reciprocating      12736
NaN                 2303
Turbo Prop           879
Turbo Fan            319
Unknown               64
Turbo Jet             41
Turbo Shaft            8
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64

 Weather.Condition
Weather.Condition
VMC    13928
NaN     1411
IMC      843
UNK      170
Name: count, dtype: int64

 Number.of.Engines
Number.of.Engines
1.0    13103
2.0     1885
NaN     1311
4.0       35
3.0       15
0.0        3
Name: count, dtype: int64

 Purpose.of.flight
Purpose.of.flight
Personal               9784
Instructional          2385
NaN                    1715
Aerial Application      719
Business                401
Positioning             254
Unknown                 243
Skydiving               151
Aerial Observation      145
Other Work Use          118
Banner Tow               86
Ferry                    71
Flight Test              70
Executive/corporate      64
Glider Tow               29
Name: c

In [368]:
#Clean weather conditions
aviation_df['Weather.Condition'] = aviation_df['Weather.Condition'].str.upper().str.strip()

aviation_df['Weather.Condition'] = aviation_df['Weather.Condition'].replace({
    'UNK': np.nan,
    'UNKNOWN': np.nan
})

In [369]:
#Standardization
aviation_df['Engine.Type'] = aviation_df['Engine.Type'].str.upper().str.strip()

#Replaceing placeholder-like values
aviation_df['Engine.Type'] = aviation_df['Engine.Type'].replace({
    'UNKNOWN': np.nan,
    'UNK': np.nan
})


#No of engines. Clean unrealistic values
aviation_df.loc[aviation_df['Number.of.Engines'] > 4, 'Number.of.Engines'] = np.nan


In [370]:
#Clean text purpose of flight
aviation_df['Purpose.of.flight'] = aviation_df['Purpose.of.flight'].str.upper().str.strip()

In [371]:
#Broad.phase.of.flight
#Clean formatting
aviation_df['Broad.phase.of.flight'] = aviation_df['Broad.phase.of.flight'].str.title().str.strip()

#Handle placeholders
aviation_df['Broad.phase.of.flight'] = aviation_df['Broad.phase.of.flight'].replace({
    'Unknown': np.nan,
    'Unk': np.nan
})


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [372]:
#Inspect Non-Null Counts

non_null_counts = aviation_df.notna().sum().sort_values(ascending=False)
non_null_counts


Event.Id                  16352
Make                      16352
Is.Destroyed              16352
Injury.Rate               16352
Serious.Fatal.Injuries    16352
Total.Passengers          16352
Total.Uninjured           16352
Total.Minor.Injuries      16352
Total.Serious.Injuries    16352
Total.Fatal.Injuries      16352
Investigation.Type        16352
Amateur.Built             16352
Model                     16352
Aircraft.Type             16352
Aircraft.Category         16352
Aircraft.damage           16352
Accident.Number           16352
Injury.Severity           16352
Event.Date                16352
Country                   16351
Location                  16349
Registration.Number       16235
FAR.Description           16195
Publication.Date          15832
Latitude                  15360
Longitude                 15356
Number.of.Engines         15041
Weather.Condition         14771
Purpose.of.flight         14637
Engine.Type               13984
Total.Injuries            13841
Report.S

In [373]:
#Identify Columns to Keep
cols_to_keep = non_null_counts[non_null_counts > 2000].index
cols_to_keep

Index(['Event.Id', 'Make', 'Is.Destroyed', 'Injury.Rate',
       'Serious.Fatal.Injuries', 'Total.Passengers', 'Total.Uninjured',
       'Total.Minor.Injuries', 'Total.Serious.Injuries',
       'Total.Fatal.Injuries', 'Investigation.Type', 'Amateur.Built', 'Model',
       'Aircraft.Type', 'Aircraft.Category', 'Aircraft.damage',
       'Accident.Number', 'Injury.Severity', 'Event.Date', 'Country',
       'Location', 'Registration.Number', 'FAR.Description',
       'Publication.Date', 'Latitude', 'Longitude', 'Number.of.Engines',
       'Weather.Condition', 'Purpose.of.flight', 'Engine.Type',
       'Total.Injuries', 'Report.Status', 'Airport.Name', 'Airport.Code',
       'Air.carrier', 'Broad.phase.of.flight'],
      dtype='object')

In [374]:
#Drop Columns Below Threshold
aviation_df = aviation_df[cols_to_keep]

In [375]:
aviation_df.shape

(16352, 36)

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [376]:
aviation_df.to_csv('AviationData_Cleaned.csv', index=False)